# 03 – Feature Engineering

## Objective
This notebook focuses on transforming the cleaned loan dataset into a model-ready format.
The goal is to apply consistent, explainable feature engineering techniques while avoiding
data leakage.

## Scope
- Final feature selection
- Handling outliers and skewed distributions
- Encoding categorical and ordinal variables
- Preparing train/validation datasets

> ⚠️ No modeling or evaluation is performed in this notebook.


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [3]:
df = pd.read_csv("data/loan_cleaned.csv")
df.shape


(887379, 25)

In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 887379 entries, 0 to 887378
Data columns (total 25 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    887379 non-null  int64  
 1   year                  887379 non-null  int64  
 2   issue_d               887379 non-null  object 
 3   emp_length_int        887379 non-null  float64
 4   home_ownership        887379 non-null  object 
 5   home_ownership_cat    887379 non-null  int64  
 6   income_category       887379 non-null  object 
 7   annual_inc            887379 non-null  int64  
 8   income_cat            887379 non-null  int64  
 9   loan_amount           887379 non-null  int64  
 10  term                  887379 non-null  object 
 11  term_cat              887379 non-null  int64  
 12  application_type      887379 non-null  object 
 13  application_type_cat  887379 non-null  int64  
 14  purpose               887379 non-null  object 
 15  

In [5]:
# Define target variable and check class balance

TARGET = "loan_condition_cat"

# Raw counts
df[TARGET].value_counts()

# Proportion (class balance)
df[TARGET].value_counts(normalize=True)


loan_condition_cat
0    0.924013
1    0.075987
Name: proportion, dtype: float64

In [6]:
# Cell [6]: Feature grouping

TARGET = "loan_condition_cat"

# Columns to drop (IDs, leakage, raw categoricals already encoded, etc.)
DROP_COLS = [
    "id",
    "loan_condition",     # raw label
    "issue_d",            # temporal leakage
]

# Numeric features
NUMERIC_COLS = [
    "year",
    "emp_length_int",
    "annual_inc",
    "loan_amount",
    "interest_rate",
    "dti"
]

# Categorical features (already label-encoded)
CATEGORICAL_COLS = [
    "home_ownership_cat",
    "income_cat",
    "term_cat",
    "application_type_cat",
    "purpose_cat",
    "interest_payment_cat",
    "grade_cat",
    "region"
]

print("Target:", TARGET)
print("Numeric features:", len(NUMERIC_COLS))
print("Categorical features:", len(CATEGORICAL_COLS))
print("Drop columns:", DROP_COLS)


Target: loan_condition_cat
Numeric features: 6
Categorical features: 8
Drop columns: ['id', 'loan_condition', 'issue_d']


In [7]:
# Cell [7]: Create feature matrix X and target vector y

# Target
y = df[TARGET]

# Features
X = df.drop(columns=DROP_COLS + [TARGET])

print("X shape:", X.shape)
print("y shape:", y.shape)

# Sanity check
X.head()


X shape: (887379, 21)
y shape: (887379,)


,year,emp_length_int,home_ownership,home_ownership_cat,income_category,annual_inc,income_cat,loan_amount,term,term_cat,...,application_type_cat,purpose,purpose_cat,interest_payments,interest_payment_cat,interest_rate,grade,grade_cat,dti,region
0,2011,10.0,RENT,1,Low,24000,1,5000,36 months,1,...,1,credit_card,1,Low,1,10.65,B,2,27.65,munster
1,2011,0.5,RENT,1,Low,30000,1,2500,60 months,2,...,1,car,2,High,2,15.27,C,3,1.00,leinster
2,2011,10.0,RENT,1,Low,12252,1,2400,36 months,1,...,1,small_business,3,High,2,15.96,C,3,8.72,cannught
3,2011,10.0,RENT,1,Low,49200,1,10000,36 months,1,...,1,other,4,High,2,13.49,C,3,20.00,ulster
4,2011,1.0,RENT,1,Low,80000,1,3000,60 months,2,...,1,other,4,Low,1,12.69,B,2,17.94,ulster


In [8]:
# Train-test split with stratification

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

# Verify class distribution
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))


Train shape: (709903, 21) (709903,)
Test shape: (177476, 21) (177476,)

Train target distribution:
loan_condition_cat
0    0.924014
1    0.075986
Name: proportion, dtype: float64

Test target distribution:
loan_condition_cat
0    0.924012
1    0.075988
Name: proportion, dtype: float64


In [9]:
# Missing value audit

missing_train = X_train.isnull().mean().sort_values(ascending=False)
missing_test = X_test.isnull().mean().sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "train_missing_pct": missing_train,
    "test_missing_pct": missing_test
})

# Show only columns with missing values
missing_summary[missing_summary > 0].dropna(how="all")


,train_missing_pct,test_missing_pct


In [10]:
# Save train-test datasets for downstream notebooks

X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Train-test datasets saved successfully.")


Train-test datasets saved successfully.


In [11]:
# Load saved train-test datasets

X_train = pd.read_csv("X_train.csv")
X_test = pd.read_csv("X_test.csv")
y_train = pd.read_csv("y_train.csv").squeeze()
y_test = pd.read_csv("y_test.csv").squeeze()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (709903, 21)
X_test: (177476, 21)
y_train: (709903,)
y_test: (177476,)


In [12]:
# Identify numeric features to scale

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

numeric_features


['year',
 'emp_length_int',
 'home_ownership_cat',
 'annual_inc',
 'income_cat',
 'loan_amount',
 'term_cat',
 'application_type_cat',
 'purpose_cat',
 'interest_payment_cat',
 'interest_rate',
 'grade_cat',
 'dti']

In [13]:
# Separate continuous numeric features from encoded categoricals

encoded_categorical_features = [
    "home_ownership_cat",
    "income_cat",
    "term_cat",
    "application_type_cat",
    "purpose_cat",
    "interest_payment_cat",
    "grade_cat"
]

continuous_numeric_features = [
    col for col in numeric_features
    if col not in encoded_categorical_features
]

continuous_numeric_features


['year', 'emp_length_int', 'annual_inc', 'loan_amount', 'interest_rate', 'dti']

In [14]:
# ============================
# Proper Feature Engineering
# ============================

from sklearn.preprocessing import StandardScaler

# 1. Start from already-split raw data
X_train_raw = X_train.copy()
X_test_raw  = X_test.copy()

# 2. Scale continuous features (fit on train only)
scaler = StandardScaler()

X_train_scaled = X_train_raw.copy()
X_test_scaled  = X_test_raw.copy()

X_train_scaled[continuous_numeric_features] = scaler.fit_transform(
    X_train_raw[continuous_numeric_features]
)

X_test_scaled[continuous_numeric_features] = scaler.transform(
    X_test_raw[continuous_numeric_features]
)

# 3. Build FINAL feature matrices
X_train_fe = pd.concat(
    [
        X_train_scaled[continuous_numeric_features],
        X_train_scaled[encoded_categorical_features]
    ],
    axis=1
)

X_test_fe = pd.concat(
    [
        X_test_scaled[continuous_numeric_features],
        X_test_scaled[encoded_categorical_features]
    ],
    axis=1
)

print("X_train_fe:", X_train_fe.shape)
print("X_test_fe:", X_test_fe.shape)


X_train_fe: (709903, 13)
X_test_fe: (177476, 13)


In [20]:
X_train_fe.select_dtypes(include=["object", "category"]).columns


Index([], dtype='object')

In [21]:
X_train_fe.to_csv("X_train_fe.csv", index=False)
X_test_fe.to_csv("X_test_fe.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("✅ Feature-engineered datasets saved correctly.")


✅ Feature-engineered datasets saved correctly.
